# Model Training (reference data are videos)

In [1]:
filename = 'ADTC_8_adz_raw_20260309'

In [2]:
from video_download import video_downloader

datum = video_downloader(filename)

Connection with geo-amberg.ch
Successful access to /cam/2026/03/09
ADTC Rueschlikon_00_20260309093021.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309083922.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309072717.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309095527.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309145144.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309060808.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309080624.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309125414.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309073927.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309101213.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309070851.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309075900.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309070251.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309065537.mp4 successfully downloaded
ADTC Rueschlikon_00_20260309075352.mp4 successfully

## DataFrame for each video (path and timestamp)

In [4]:
import os
import pandas as pd
from pathlib import Path

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'VIDEO/{datum}'

rows = []

for f in os.listdir(src):
    file = os.path.join(src, f)
    t = f[-18:-1]
    year = t[0:4]
    month = t[4:6]
    day = t[6:8]
    hour = t[8:10]
    minute = t[10:12]
    seconde = t[12:14]

    timestamp_video = f'{year}-{month}-{day} {hour}:{minute}:{seconde}'
    
    rows.append({
        "video": file,
        "timestamp_video": pd.to_datetime(timestamp_video)
    })

df_video = pd.DataFrame(rows)
df_video.head()

,video,timestamp_video
0,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 00:06:08
1,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 00:07:32
2,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 00:22:19
3,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 00:32:06
4,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 00:36:25


In [17]:
df_train_data = pd.read_csv(f'DataFrame/df_merged_{filename}.csv')

df_video['timestamp_video'] = pd.to_datetime(df_video['timestamp_video'])
df_train_data['timestamp'] = pd.to_datetime(df_train_data['timestamp'])

df_video_merged = pd.merge_asof(
    df_video.sort_values('timestamp_video'),
    df_train_data.sort_values('timestamp'),
    left_on='timestamp_video',
    right_on='timestamp',
    direction='nearest',
    tolerance=pd.Timedelta('10s')
)

df_video_merged = df_video_merged.dropna(subset=['file'])

cols = [c for c in df_video_merged.columns if c not in ["video", "timestamp_video"]] + ["video", "timestamp_video"]
df_video_merged = df_video_merged[cols]

df_video_merged.head(5)

,file,timestamp_raw,timestamp,timestamp_serie,delta_t,velocity,length,total_length,first_peak_raw,video,timestamp_video
1,ADTC_8_adz_raw_20260309_0010,2026-03-09 00:07:20.748,2026-03-09 00:07:30.800,"[Timestamp('2026-03-09 00:07:30.800000'), Time...","[0.127, 0.863, 0.128, 0.254, 0.137, 0.907, 0.1...","[18.713734729162315, 18.533094812164578, 18.02...","[2.376644310603614, 15.99406082289803, 2.30770...",96.077610,9.916599,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 00:07:32
2,ADTC_8_adz_raw_20260309_0030,2026-03-09 00:22:07.505,2026-03-09 00:22:17.540,"[Timestamp('2026-03-09 00:22:17.540000'), Time...","[0.11, 0.762, 0.114, 0.215, 0.122, 0.781, 0.12...","[21.24268118354319, 21.014714898835074, 20.757...","[2.336694930189751, 16.013212752912327, 2.3664...",96.789062,9.916599,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 00:22:19
5,ADTC_8_adz_raw_20260309_0310,2026-03-09 03:02:50.168,2026-03-09 03:03:00.355,"[Timestamp('2026-03-09 03:03:00.355000'), Time...","[0.12, 0.233, 0.121, 0.198, 0.103, 0.513, 0.10...","[22.787376388169307, 22.677376171352073, 22.69...","[2.7344851665803165, 5.283828647925033, 2.7464...",178.766932,9.916599,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 03:03:01
6,ADTC_8_adz_raw_20260309_0450,2026-03-09 04:44:06.406,2026-03-09 04:44:16.600,"[Timestamp('2026-03-09 04:44:16.600000'), Time...","[0.11, 0.72, 0.099, 0.208, 0.103, 0.716, 0.107...","[22.09439059272207, 22.155969163933726, 22.561...","[2.4303829651994278, 15.952297798032282, 2.233...",96.749232,9.916599,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 04:44:18
7,ADTC_8_adz_raw_20260309_0510,2026-03-09 05:00:24.483,2026-03-09 05:00:34.738,"[Timestamp('2026-03-09 05:00:34.738000'), Time...","[0.135, 0.908, 0.133, 0.254, 0.143, 0.906, 0.1...","[17.696228338430174, 17.476808905380334, 17.66...","[2.3889908256880736, 15.868942486085345, 2.349...",96.397686,9.916599,c:\git\RailwAI\VIDEO\20260309\ADTC Rueschlikon...,2026-03-09 05:00:36


In [28]:
import csv
import os

# change to the file directory
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

src = root / f'VIDEO/{datum}/'

rest = []

for f in os.listdir(src):
    if f == 'reference_video.csv':
        continue
    file = os.path.join(src, f)

    if file not in df_video_merged['video'].values:
        os.remove(file)
    else:
        rest.append(file)
    
output_csv = os.path.join(src, 'reference_video.csv')

if os.path.exists(output_csv):
    print('reference_video.csv already exists')
    
else:
    with open(output_csv, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=';')  # use ; as separator
        writer.writerow(['video', 'train_type'])
        for vid in rest:
            writer.writerow([vid, ''])

reference_video.csv already exists


In [29]:
d = Path.cwd()
if d == Path('C:/git/RailwAI/code'):
    root = d.parent
elif d == Path('C:/git/RailwAI'):
    root = d
else:
    print('ERROR: you are actually in ' + str(Path.cwd()))

csv_train = root / f'VIDEO/{datum}/reference_video.csv'
df_ref = pd.read_csv(csv_train, sep=';')
df_video_merged = df_video_merged.merge(df_ref, on='video', how='left')